## Prompt Optimisation with MLflow

### Installing Utilities and Libraries

In [ ]:
%pip install mlflow==3.15.2 openai==2.38.0

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# loading azure openai configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
chat_completions_model_name = os.getenv("CHAT_COMPLETIONS_MODEL_NAME")

### Enable MLflow Tracing

In [ ]:
import mlflow

# Calling autolog for LangChain will enable trace logging.
mlflow.openai.autolog()

# Optional: Set a tracking URI and an experiment
mlflow.set_experiment("prompt-optimization")
mlflow.set_tracking_uri("http://localhost:5000")

### Register the Initial Prompt

In [ ]:
PROMPT_NAME = "ticket-classification-prompt"

prompt = mlflow.genai.register_prompt(
    name=PROMPT_NAME,

    template="""
Classify the following IT support ticket.

Ticket:
{{ticket}}
""",

    commit_message="v1: Basic ticket classification prompt"
)


# set a production alias
mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="development",
    version=1
)

print(f"Prompt: {prompt.name}")
print(f"Version: {prompt.version}")

### Define the Prediction Function

In [ ]:
from openai import AzureOpenAI

def predict_fn(query):

        prompt = mlflow.genai.load_prompt(
            name_or_uri=(
                f"prompts:/{PROMPT_NAME}@development"
            )
        )

        formatted_prompt = prompt.format(
            ticket=query
        )

        # creating the Azure OpenAI client to process user requests
        azure_openai_client = AzureOpenAI(
            api_key=azure_openai_api_key,
            api_version="2024-02-15-preview",
            azure_endpoint=azure_openai_endpoint
        )
        
        # sending a chat completions request to the LLM
        chat_completions_response = azure_openai_client.chat.completions.create(
            model = chat_completions_model_name,
            messages = [
                {"role": "system", "content": "You are a helpful AI assistant"},
                {"role": "user", "content": formatted_prompt}
            ],
            temperature=0.7
        )
    
        return chat_completions_response.choices[0].message.content

### Test your Prediction Function

In [ ]:
ticket_query = "My second monitor occasionally flickers, but I can continue working normally"

output = predict_fn(query = ticket_query)

print(output)

### Optimise against Data

In [ ]:
# Training data with inputs, expected outputs, and expectations
dataset = [
    {
        "inputs": {
            "query": (
                "I changed my password this morning and "
                "now I cannot log into my corporate account."
            )
        },
        "outputs": {
            "response": "Category: Authentication\nPriority: High"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Authentication'",
                "Priority must be 'High'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "The office Wi-Fi is unavailable for everyone "
                "on the third floor and nobody can access "
                "internal applications."
            )
        },
        "outputs": {
            "response": "Category: Network\nPriority: Critical"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Network'",
                "Priority must be 'Critical'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "Microsoft Excel crashes whenever I try to "
                "open one particular spreadsheet. Other "
                "spreadsheets work normally."
            )
        },
        "outputs": {
            "response": "Category: Software\nPriority: Medium"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Software'",
                "Priority must be 'Medium'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "I received an email asking me to enter my "
                "company password on an unfamiliar website."
            )
        },
        "outputs": {
            "response": "Category: Security\nPriority: High"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Security'",
                "Priority must be 'High'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "My second monitor occasionally flickers, "
                "but I can continue working normally."
            )
        },
        "outputs": {
            "response": "Category: Hardware\nPriority: Low"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Hardware'",
                "Priority must be 'Low'"
            ]
        }
    }
]

In [ ]:
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Correctness

# Optimize the prompt
result = mlflow.genai.optimize_prompts(
    predict_fn = predict_fn,
    train_data = dataset,
    prompt_uris = [prompt.uri],
    optimizer = GepaPromptOptimizer(reflection_model = "anthropic:/claude-opus-4-5"),
    scorers = [Correctness(model="anthropic:/claude-opus-4-5")]
)

# Use the Optimized Prompt
optimized_prompt = result.optimized_prompts[0]
print(f"Optimized Template: {optimized_prompt.template}")

### Use the Optimised Prompt

In [ ]:
VERSION_NUMBER = "LATEST_PROMPT_VERSION_NUMBER"

def predict_fn(query):

        prompt = mlflow.genai.load_prompt(
            name_or_uri=(
                f"prompts:/{PROMPT_NAME}/{VERSION_NUMBER}"
            )
        )

        formatted_prompt = prompt.format(
            ticket=query
        )

        # creating the Azure OpenAI client to process user requests
        azure_openai_client = AzureOpenAI(
            api_key=azure_openai_api_key,
            api_version="2024-02-15-preview",
            azure_endpoint=azure_openai_endpoint
        )
        
        # sending a chat completions request to the LLM
        chat_completions_response = azure_openai_client.chat.completions.create(
            model = chat_completions_model_name,
            messages = [
                {"role": "system", "content": "You are a helpful AI assistant"},
                {"role": "user", "content": formatted_prompt}
            ],
            temperature=0.7
        )
    
        return chat_completions_response.choices[0].message.content

In [ ]:
ticket_query = "My second monitor occasionally flickers, but I can continue working normally"

output = predict_fn(query = ticket_query)

print(output)